# Generating Neutral Redistricting Maps for Utah

Welcome! This tool is designed to empower you to explore fair and neutral possibilities for Utah's electoral districts. You will be using a powerful method called ensemble analysis to generate thousands of potential district maps that all follow a consistent set of neutral rules based on Utah's Proposition 4.

## Background
### What is an Ensemble and Why Is It Useful?

It's impossible to find the single "best" or "fairest" map. Instead, we can generate a large collection of maps called an __ensemble__ that all comply with a given set of legal and community-based criteria.

Think of it like this: if you ask 10,000 different people to draw a "fair" map, you'll (hopefully) get 10,000 different valid maps. By studying this entire collection, we can understand the _range_ of possible outcomes. If an officially proposed or enacted map looks very different from the vast majority of maps in our neutral ensemble, it might be an outlier worth scrutinizing.

To create this ensemble, we use a technique called a __Markov Chain Monte Carlo (MCMC)__ simulation. Specifically, we use a method called __Recombination (ReCom)__, which is designed to explore a wide variety of map configurations efficiently. In simple terms, our program will:
1. Start with an existing map.
2. Randomly pick two neighboring districts.
3. Temporarily merge them into one large "super-district".
4. Randomly draw a new boundary within this super-district to split it back into two new, population-balanced districts.
5. Check if this newly-drawn map follows all of our other rules.
6. If it does, we add it to our collection and repeat.

By doing this thousands of times, we build a massive, diverse ensemble of valid district maps.

### Utah's Neutral Redistricting Standards

Utah law provides the following standards for how maps must be drawn, which are given an order of priority.
1. Follow the US. Constitution and federal law, maintaining equal population in districts and complying with the Voting Rights Act.
2. Minimize splitting cities and counties, with cities being most important to keep whole.
3. Make districts geographically compact.
4. Make districts contiguous and allow ease of travel throughout district
5. Preserve neighborhoods and communities of interest
6. Where possible, follow mountains, rivers, and other natural boundaries.
7. Where possible, line up the boundaries of different types of districts (e.g., Congressional and State Senate)
(Adapted from [Better Boundaries](https://betterboundaries.org/2025/09/21/guide-to-evaluating-maps/]))

We have a variety of tools at our disposal to help the computer follow these rules, but how _exactly_ to balance the different priorities is a question for humans to answer.

## Select parameters
In the sections below, you can configure different parameters to match your interpretation of the law's priorities. There's no harm in experimenting, so give it a go!

In [ ]:
# Configuration files and example maps will be saved to a directory with the current date and time plus an optional user-defined tag
config_tag = "transitability_test"  # <-- Change this to something descriptive if desired

### Constraints
One way to guide map drawing is through __constraints__. When a constraint is not met, the map is rejected and the algorithm must draw new maps until it creates one that passes. 

In [ ]:
# Constraints configuration
constraint_params = {
    # How much can a district's population deviate from the ideal population?
    "pop_deviation": 0.001,  # 0.001 is equivalent to 0.1%
    # How many municipalities can be split total?
    "split_munis_constraint": 3,
    # How many splits past the first for each muni are allowed (map wide)?
    "muni_multi_splits_constraint": 0,
    # How many counties can be split total?
    "split_counties_constraint": 3,
    # How many splits past the first for each county are allowed (map wide)?
    "county_multi_splits_constraint": 0,
}

### Region surcharges
A __region surcharge__ adjusts the probability that the algorithm will draw a line that cuts through a region of that type, with _higher_ values making it _less_ likely to be split. Using all zeros would allow the algorithm to draw lines with uniform probablity--leaving it very free to explore, but potentially very bad at meeting constraints or honoring that particular criterion. On the other hand, large values help encourage keeping that region type intact (and meeting any related constraints). If a surcharge is too large, however, it can dramatically limit the space of maps that are possible to explore, and the algorithm might draw the same map many times. 

### Transitability Graph
You can optionally provide a precomputed transitability graph to use instead of building it from scratch. This can significantly speed up the analysis if you have already computed the transitability-aware connectivity.


In [ ]:
 # Transitability graph configuration
transitability_graph = "data/transitability/_waterborders/transitability.json"  # Path to precomputed graph

In [ ]:
# Region surcharge configuration
region_surcharge_params = {
    "muni": 2,  # municipalities
    "county": 1,  # counties
    "highered": 0.5,  # institutions of higher education
    "metro": 0.5,  # metro/micropolitan areas
    "school_district": 0.1,  # school districts
    "water_region": 0.1,  # water planning regions
    "basin": 0.1,  # hydrologic basins
}

### Tilted run
The last way to influence the map drawing process is through a __tilted run__. During a tilted run, the algorithm scores each map on some criterion. If a map scores better than the one before it, the map is accepted. If the map scores worse, there is some probability that the map will be rejected. This encourages the ensemble generally to sample maps that perform well on that score, without enforcing it strictly. Here, we will use it to encourage compact districts.

In [ ]:
# Tilted run configuration
tilted_run_params = {
    # Probability of accepting a map less compact than the previous map
    "less_compact_probability": 1.0,  
}

### Other settings
Choose how many maps this notebook will generate and how many should be rendered for visualization.

You can also choose a "seed" for the random number generator, so your exact ensemble can be reproduced later.

In [ ]:
# Initialization parameters
init_params = {
    "random_seed": 1847,  # Seed for the random number generator
    # Path to an shapefile that will be used to initialize the algorithm
    "initial_partition": "plans/CONG/2025_UT-C/2025_UT-C.shp",
    # Path to a shapefile that will be used to initialize the nodes
    "nodes_data": "data/UT_precincts.geojson",
}
# Ensemble parameters
ensemble_params = {
    "steps": 51,  # Number of maps to generate
    "visualize_every": 1,  # Number of maps between renders
}

### Transitability
Transitability analysis ensures that districts are connected by actual roads and not separated by impassable barriers like major water bodies. This implements Utah's requirement for "ease of travel throughout district" by:

1. **Road Connectivity**: Verifying that precincts are connected by actual road networks
2. **Hierarchical Fallback**: For rural areas with only local roads, fall back to county boundaries
3. **Water Barriers**: Remove connections that cross major water bodies (Great Salt Lake, Lake Powell, etc.)

This analysis respects the priority of keeping municipalities and counties whole while ensuring physical connectivity.


In [ ]:
# Transitability configuration
transitability_params = {
    "enable": True,  # Enable transitability analysis
    "remove_water_barriers": True,  # Remove edges crossing major water bodies
    "verify_road_connectivity": True,  # Verify road network connectivity
    "min_lake_size_sqkm": 1.0,  # Minimum lake size to consider as barrier
    "min_river_size_sqkm": 0.5,  # Minimum river size to consider as barrier
    "road_buffer_meters": 500,  # Buffer distance for road-precinct intersection
    "water_threshold": 0.5,  # Minimum fraction of edge crossing water to remove
}


### Preconditioning
Before running the ensemble, modify the starting map to comply meet all constraints and tuning it to be an approximately 'neutral' map under the parameters you have defined. Also identifies if any parameters might need to be adjusted in order to successfully create an ensemble.

In [ ]:
# Preconditioning configuration
preconditioning_params = {
    "enable": True,
    "steps": 20,  # Number of maps to draw during each preconditioning run
    # If the constraints are not met after a preconditioning run, preconditioning will be repeated until either the constraints are met or the maximum number of repeats is reached.
    "max_repeats": 10,
}
# Election parameters need to be configured, but are not used for neutral sampling inside this notebook. Kept as a placeholder/reminder for future work.
# election_params = {
#     "vote_share_agg": "median",
#     "years": "2016,2020,2024",
#     "offices": "PRE,GOV,ATG,AUD,TRE",
# }

In [ ]:
# Combine all config parameters into a single configuration dictionary
config = {
    "constraints": constraint_params,
    "region_surcharges": region_surcharge_params,
    "tilted_run": tilted_run_params,
    "initialization": init_params,
    "ensemble": ensemble_params,
    "preconditioning": preconditioning_params,
    "transitability": transitability_params,
}

In [ ]:
import os
from datetime import datetime

# Create a directory name using the current date and the tag
today_str = datetime.now().strftime("%Y%m%d%H%M%S")
if config_tag:
    config_dir = f"results/configurations/{today_str}_{config_tag}"
else:
    config_dir = f"results/configurations/{today_str}"

# Ensure the directory exists
os.makedirs(config_dir, exist_ok=True)

In [ ]:
from utgc.ensemble import EnsembleRunner

# Create and run ensemble analysis
runner = EnsembleRunner(config, transitability_graph=transitability_graph)
results = runner.run(output_dir=config_dir)

print(f"Ensemble analysis complete! Generated {len(results)} maps.")
print(f"Results saved to: {config_dir}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rows = []
for r in results:
    step = r.get("step")
    split_m = r.get("split_munis_count")
    split_c = r.get("split_counties_count")
    split_m_multi = r.get("split_munis_extra_parts", 0)
    split_c_multi = r.get("split_counties_extra_parts", 0)
    cut_e = r.get("num_cut_edges")
    pop_dev = r.get("population_deviation", {})
    try:
        dev_vals = [v for v in pop_dev.values() if v is not None]
        max_dev = max(dev_vals) if dev_vals else None
        mean_dev = sum(dev_vals) / len(dev_vals) if dev_vals else None
    except Exception:
        max_dev = None
        mean_dev = None
    rows.append({
        "step": step,
        "split_munis_count": split_m,
        "split_counties_count": split_c,
        "split_munis_multi": split_m_multi,
        "split_counties_multi": split_c_multi,
        "population_max_dev": max_dev,
        "population_mean_dev": mean_dev,
        "num_cut_edges": cut_e,
    })

metrics_df = pd.DataFrame(rows).sort_values("step")

sns.set_style("whitegrid")
fig, axes = plt.subplots(4, 1, sharex=True)

axes[0].plot(metrics_df["step"], metrics_df["population_max_dev"], label="Max pop dev", color="#1f77b4")
axes[0].plot(metrics_df["step"], metrics_df["population_mean_dev"], label="Mean pop dev", color="#ff7f0e")
axes[0].set_ylabel("Pop. dev.")

# Plot municipality splits
if "split_munis_count" in metrics_df.columns:
    axes[1].plot(metrics_df["step"], metrics_df["split_munis_count"], label="Muni splits", color="#2ca02c")
if "split_munis_multi" in metrics_df.columns:
    axes[1].plot(metrics_df["step"], metrics_df["split_munis_multi"], label="Muni multi-splits", color="#2ca02c", linestyle="--")
axes[1].set_ylabel("Muni splits")
axes[1].legend()

# Plot county splits
if "split_counties_count" in metrics_df.columns:
    axes[2].plot(metrics_df["step"], metrics_df["split_counties_count"], label="County splits", color="#d62728")
if "split_counties_multi" in metrics_df.columns:
    axes[2].plot(metrics_df["step"], metrics_df["split_counties_multi"], label="County multi-splits", color="#d62728", linestyle="--")
axes[2].set_ylabel("County splits")
axes[2].legend()

axes[3].plot(metrics_df["step"], metrics_df["num_cut_edges"], label="Cut edges", color="#6b7280")
axes[3].set_ylabel("Cut edges")
axes[3].set_xlabel("Step")

fig.set_dpi(300)
plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import display
import ipywidgets as widgets
from PIL import Image

# Load PNGs that begin with "step_" in the config_dir directory
image_dir = str(config_dir)
image_files = sorted(
    [f for f in os.listdir(image_dir) if f.lower().endswith(".png") and f.startswith("step_")]
)

img = widgets.Image(format='png')

steps = list(range(0, ensemble_params["steps"], ensemble_params["visualize_every"]))
steps2index = {s: i for i, s in enumerate(steps)}
# Slider shows actual step numbers
stepper = widgets.BoundedIntText(value=steps[0], min=min(steps), max=max(steps), step=ensemble_params["visualize_every"], description="Step:")

frames = []
for fname in image_files:
    with open(os.path.join(image_dir, fname), "rb") as f:
        data = f.read()
    frames.append(data)

# Ensure no duplicate observers if you re-run this cell
try:
    stepper.unobserve_all()
except Exception: pass

def on_change(value):
    img.value = frames[steps2index[value]]

widgets.interactive(on_change, value=stepper)
img.value = frames[steps2index[stepper.value]]

# Create and display the widget
widget_box = widgets.VBox([stepper, img])
display(widget_box)

In [ ]:
# Export YAML configuration for CLI runs inside the tag folder
import yaml

yaml_path = os.path.join(config_dir, "params.yaml")
with open(yaml_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=True)

print(f"Wrote YAML: {yaml_path}")